# Lab 2: Goal-Based Agent for Warehouse Navigation

**Name:** Mehta Prem Kalpesh 

**ID:** 2024ADPS0851G

## Task 1: Understanding the Problem

### 1. What is the environment?
The environment is the warehouse grid: a 2D map of free cells (`.`), obstacles (`#`), the start cell `S` (loading bay) and the goal cell `G` (dispatch area). It has the following properties:

| Property | Value | Reason |
|---|---|---|
| Observability | Fully observable | The whole map is known to the agent |
| Determinism | Deterministic | A move always has the intended effect |
| Dynamics | Static | Obstacles never move or change |
| State/action space | Discrete | Finite grid cells and four moves |
| Episodes | Sequential | Each move affects later moves |
| Agents | Single-agent | Only one vehicle is considered |

### 2. What is the goal of the agent?
The goal is to reach the goal state `G` from the start state `S` through a sequence of legal moves, avoiding all obstacles. Every move costs 1, so the best solution is the path with the fewest moves.

### 3. What actions are available to the agent?
At each step the agent can move **Up, Down, Left or Right** by one grid cell. A move is legal only if the target cell is inside the warehouse boundary and is not an obstacle (`#`).

### 4. What information must the agent maintain?
- **Current position** (the current state)
- **Goal position**, to test whether the goal has been reached
- **The map**, i.e. the agent's model of the environment, used to check neighbouring cells for obstacles and boundaries
- **Front cells**, discovered cells not yet explored
- **Visited path**, so that no cell is visited twice
- **Parent pointers**, to reconstruct the route once `G` is reached

### 5. Why is this a goal-based agent rather than a simple reflex agent?
A simple reflex agent chooses actions using only the **current percept** through condition-action rules (e.g. "if wall ahead, turn right"). It has no explicit goal and does not look ahead. In this maze it can get trapped in dead ends or loop forever, and it has no guarantee of reaching `G`.

A goal-based agent has an **explicit goal** and chooses actions by **considering sequences of future states** (search) to find a route that reaches it. This lets it avoid dead ends and guarantees it finds a path if one exists. Simply adding memory of visited cells to a reflex agent would make it a model-based agent, but it would still lack the goal and the lookahead.

### Think About It: if the warehouse becomes twice as large
Doubling both dimensions gives about **4× as many cells**. A breadth-first search (BFS) still finds the shortest path, but it explores cells in all directions, so its time and memory (frontier + visited set) grow with the area explored, and this can become expensive.

A better choice is **A\*** with the Manhattan-distance heuristic. The heuristic is admissible on a 4-direction grid, so A\* remains optimal while expanding far fewer cells by steering toward `G`.

Further difficulties that could arise:
- Moving obstacles or other vehicles (dynamic environment)
- Incomplete knowledge of the map (partial observability)
- The need to replan while executing the path
- Real-time constraints on how long the agent can spend planning

## Task 2: Designing the Agent

### Design components

| Component | Specification |
|---|---|
| **Environment** | The warehouse as a 2D grid of rows and columns containing free cells (`.`), obstacles (`#`), a start cell `S` and a goal cell `G`. Fully observable, deterministic, static, discrete. |
| **Current state** | The agent's position in the grid as `(row, column)`. The start state is the position of `S`. |
| **Goal** | The goal state is the position of `G`. The goal test is: *current state == goal position*. When it succeeds, the path from `S` to this state is returned as the solution. |
| **Available actions** | Move **Up, Down, Left, Right** by one cell. An action is legal only if the resulting cell is inside the grid boundary and is not an obstacle (`#`). |
| **Decision-making component** | A search procedure. It keeps a **frontier** (discovered cells not yet explored) and a **visited set**. It repeatedly takes a cell from the frontier, applies the goal test, and otherwise tries all four moves. Each move that is legal (in bounds, not `#`) and leads to an unvisited cell is added to the frontier with a record of its parent cell. On reaching `G`, the path is rebuilt from the parent records. If the frontier becomes empty, no path exists. |

### Block diagram

```
 +--------------------+                      +----------------------+
 |    WORLD MODEL     |                      |         GOAL         |
 |  grid, obstacles,  |                      |  goal state G        |
 |  boundary, move    |                      |  goal test:          |
 |  rules             |                      |  state == G ?        |
 +---------+----------+                      +-----------+----------+
           |                                             |
           +----------------------+----------------------+
                                  |
                                  v
 +------------------------------------------------------------------+
 |             DECISION-MAKING COMPONENT (search)                   |
 |                                                                  |
 |   frontier  ->  pick a cell  ->  goal test (state == G ?)        |
 |                      |                                           |
 |        try Up / Down / Left / Right                              |
 |                      |                                           |
 |        legal move? (in bounds, not '#', not visited)             |
 |             yes -> add to frontier, store parent                 |
 |                                                                  |
 |   goal reached -> rebuild path | frontier empty -> no path       |
 +---------------+--------------------------------+-----------------+
                 ^                                |
                 | current state                  | chosen action
                 | (row, col)                     | (next move)
         +-------+--------+               +-------v--------+
         |    SENSORS     |               |   ACTUATORS    |
         |  read position |               |  move vehicle  |
         |  + surroundings|               |  one cell      |
         +-------+--------+               +-------+--------+
                 ^                                |
                 | percept                        | action
                 |                                v
 +------------------------------------------------------------------+
 |                 ENVIRONMENT (warehouse grid)                     |
 |           free cells '.', obstacles '#', start S, goal G         |
 +------------------------------------------------------------------+
```

The agent senses its position, and the search component combines it with the world model and the goal to look ahead and choose a sequence of moves. The actuators then execute the next move, which changes the vehicle's position in the environment, and the cycle repeats. Unlike a simple reflex agent, the percept does not map directly to an action. The decision is made by searching over future states.

## Task 3: Prompt Engineering

### Prompt given to the LLM

```text
Write a well-documented Python 3 program implementing a goal-based agent for the warehouse navigation problem below.

PROBLEM
An autonomous warehouse vehicle must move from the loading bay (S) to the dispatch area (G) along a collision-free path. The warehouse is this map, where S = start, G = goal, # = obstacle, . = free space:

#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################

AGENT DESIGN (please follow this design)
- Environment: the warehouse as a 2D grid of rows and columns containing free cells, obstacles, a start cell S and a goal cell G. It is fully observable, deterministic, static and discrete.
- Current state: the agent's position in the grid as (row, column). The start state is the position of S.
- Goal: the goal state is the position of G. The goal test is: current state == goal position. When the test succeeds, the path from S to that state is returned as the solution.
- Available actions: move Up, Down, Left or Right by one cell. An action is legal only if the resulting cell is inside the grid boundary and is not an obstacle (#).
- Decision-making component: a search procedure that keeps a frontier (discovered cells not yet explored) and a visited set. It repeatedly takes a cell from the frontier, applies the goal test, and otherwise tries all four moves. Each legal move to an unvisited cell is added to the frontier with a record of its parent cell. On reaching G, the path is rebuilt from the parent records. If the frontier becomes empty, no path exists.

REQUIREMENTS
- Represent the warehouse as a two-dimensional grid (a list of lists or a list of strings) and locate S and G by scanning it.
- Use (row, column) coordinates, 0-indexed.
- Determine a collision-free path from S to G that avoids all obstacles and never leaves the grid.
- Print either the path found (as a list of (row, column) coordinates and as a sequence of moves such as Right, Down, ...) or a clear message if no path exists. Also print the number of moves.
- Also print the warehouse with the path marked by '*' (keeping S and G visible).
- Explain, in comments and in the printed output, which search algorithm was chosen and why it is appropriate for this problem.
- Structure the code into small, documented functions (e.g. parse the grid, find S and G, get legal moves, search, reconstruct the path, display the result) with docstrings, and a main() function.
- Use only the Python standard library, and make the program run with no user input.
- Include a second test where the goal is walled off, to show that the "no path" message works.
```

### Observations

1. **Did the LLM generate a working program on the first attempt?**
   Mostly: the search is correct, but the unreachable test case didn’t actually block the goal. I fixed it by changing the test case to properly block the path to G.

2. **If not, how can the prompt be improved?**
   The second test map must make G completely unreachable, and the program should print the no-path message. Hence, it should assert the expected result in each test.

3. **What search algorithm did the LLM choose?**
   Breadth-First Search

4. **Why do you think the LLM selected this algorithm?**
   The grid is unweighted (each move costs 1), BFS guarantees the shortest path, and the state space is small.

In [5]:
"""
Goal-Based Warehouse Navigation Agent
-------------------------------------
This module implements a goal-based agent that finds an optimal (shortest)
collision-free path from a loading bay (S) to a dispatch area (G) in a 2D warehouse grid.

Search Algorithm Selection:
---------------------------
Breadth-First Search (BFS) is chosen for this problem because:
1. The grid is unweighted (each move has a uniform cost of 1).
2. BFS guarantees finding the shortest optimal path in unweighted graphs.
3. The state space (grid cells) is finite and small, so memory consumption is negligible.
4. BFS systematically explores all nodes at depth 'k' before moving to 'k+1', ensuring
   optimality without requiring complex heuristics.
"""

from collections import deque
from typing import List, Tuple, Dict, Optional

# Movement vectors and corresponding action names
MOVES: Dict[str, Tuple[int, int]] = {
    "Up": (-1, 0),
    "Down": (1, 0),
    "Left": (0, -1),
    "Right": (0, 1),
}


def parse_grid(raw_map: str) -> List[List[str]]:
    """Converts a multi-line string representation of a map into a 2D grid (list of lists).

    Args:
        raw_map: Multi-line string containing the grid elements.

    Returns:
        List of lists of strings representing grid rows and columns.
    """
    lines = [line.strip() for line in raw_map.strip().splitlines() if line.strip()]
    return [list(line) for line in lines]


def find_positions(grid: List[List[str]]) -> Tuple[Tuple[int, int], Tuple[int, int]]:
    """Scans the 2D grid to locate the Start (S) and Goal (G) coordinates.

    Args:
        grid: 2D list representing the warehouse layout.

    Returns:
        A tuple of (start_pos, goal_pos) where each is a (row, column) tuple.

    Raises:
        ValueError: If 'S' or 'G' is not found in the grid.
    """
    start: Optional[Tuple[int, int]] = None
    goal: Optional[Tuple[int, int]] = None

    for r, row in enumerate(grid):
        for c, cell in enumerate(row):
            if cell == "S":
                start = (r, c)
            elif cell == "G":
                goal = (r, c)

    if start is None or goal is None:
        raise ValueError("Grid must contain both a Start ('S') and Goal ('G') symbol.")

    return start, goal


def get_legal_moves(
    grid: List[List[str]], pos: Tuple[int, int]
) -> List[Tuple[Tuple[int, int], str]]:
    """Determines all valid, collision-free moves from a given position.

    Args:
        grid: 2D list representing the warehouse map.
        pos: Current position (row, col).

    Returns:
        A list of tuples ((next_row, next_col), move_name) for all valid moves.
    """
    rows = len(grid)
    cols = len(grid[0])
    r, c = pos
    legal = []

    for move_name, (dr, dc) in MOVES.items():
        nr, nc = r + dr, c + dc
        # Check boundary bounds and obstacle collision
        if 0 <= nr < rows and 0 <= nc < cols:
            if grid[nr][nc] != "#":
                legal.append(((nr, nc), move_name))

    return legal


def bfs_search(
    grid: List[List[str]], start: Tuple[int, int], goal: Tuple[int, int]
) -> Tuple[Optional[List[Tuple[int, int]]], Optional[List[str]]]:
    """Executes a Goal-Based Agent search using Breadth-First Search (BFS).

    Maintains a frontier queue and visited set. Reconstructs parent links
    upon reaching the goal.

    Args:
        grid: 2D grid of warehouse.
        start: Starting position tuple (row, col).
        goal: Target goal position tuple (row, col).

    Returns:
        A tuple (path, actions):
        - path: List of (row, col) tuples from start to goal (or None if no path exists).
        - actions: List of action names ("Up", "Down", etc.) executed along the path.
    """
    # Frontier holds (current_position)
    frontier: deque = deque([start])
    visited = {start}

    # parent mapping: child_pos -> (parent_pos, action_taken)
    parent: Dict[Tuple[int, int], Tuple[Tuple[int, int], str]] = {}

    while frontier:
        curr = frontier.popleft()

        # Goal Test
        if curr == goal:
            # Reconstruct path and action sequence
            path = [curr]
            actions = []
            while curr != start:
                prev_pos, action = parent[curr]
                actions.append(action)
                path.append(prev_pos)
                curr = prev_pos
            path.reverse()
            actions.reverse()
            return path, actions

        # Explore neighbors
        for next_pos, action in get_legal_moves(grid, curr):
            if next_pos not in visited:
                visited.add(next_pos)
                parent[next_pos] = (curr, action)
                frontier.append(next_pos)

    # No path found
    return None, None


def display_map_with_path(
    grid: List[List[str]], path: Optional[List[Tuple[int, int]]]
) -> str:
    """Renders the grid as a string, overlaying '*' on the path (preserving S and G).

    Args:
        grid: Original 2D grid.
        path: List of coordinates representing the solution path.

    Returns:
        Multi-line string representation of the grid showing the route.
    """
    # Create a deep copy of the grid for display
    display = [row[:] for row in grid]

    if path:
        for r, c in path:
            if display[r][c] not in ("S", "G"):
                display[r][c] = "*"

    return "\n".join("".join(row) for row in display)


def run_warehouse_agent(test_name: str, raw_map: str) -> None:
    """Executes the agent workflow on a given map and prints formatted results.

    Args:
        test_name: Label/Description for the test case.
        raw_map: String representation of the warehouse grid.
    """
    print("=" * 60)
    print(f"RUNNING TEST: {test_name}")
    print("=" * 60)

    grid = parse_grid(raw_map)
    start, goal = find_positions(grid)

    print(f"Start Position (S): {start}")
    print(f"Goal Position  (G): {goal}")
    print("\nSearch Algorithm Selected: Breadth-First Search (BFS)")
    print("Reason: Guarantees the shortest path in an unweighted 2D grid.")
    print("-" * 60)

    path, actions = bfs_search(grid, start, goal)

    if path and actions:
        print("RESULT: Path Found!")
        print(f"Total Moves: {len(actions)}")
        print(f"\nSequence of Moves:\n  {' -> '.join(actions)}")
        print(f"\nCoordinate Path:\n  {path}")
        print("\nWarehouse Map with Solution Path ('*' = path):")
        print(display_map_with_path(grid, path))
    else:
        print("RESULT: No path exists to the goal!")
        print("\nWarehouse Map (Unreachable Goal):")
        print(display_map_with_path(grid, None))

    print("\n")


def main() -> None:
    # Test Case 1: Standard Warehouse Map
    map_standard = """
#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################
"""

    # Test Case 2: Walled-off Goal (No Path Exists)
    map_unreachable = """
#####################
#S....#...........#G#
#.##....##########.##
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################
"""

    run_warehouse_agent("Standard Map (Valid Path)", map_standard)
    run_warehouse_agent("Blocked Goal Map (Unreachable)", map_unreachable)


if __name__ == "__main__":
    main()

RUNNING TEST: Standard Map (Valid Path)
Start Position (S): (1, 1)
Goal Position  (G): (1, 19)

Search Algorithm Selected: Breadth-First Search (BFS)
Reason: Guarantees the shortest path in an unweighted 2D grid.
------------------------------------------------------------
RESULT: Path Found!
Total Moves: 20

Sequence of Moves:
  Right -> Right -> Right -> Down -> Right -> Right -> Right -> Up -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right

Coordinate Path:
  [(1, 1), (1, 2), (1, 3), (1, 4), (2, 4), (2, 5), (2, 6), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (1, 16), (1, 17), (1, 18), (1, 19)]

Warehouse Map with Solution Path ('*' = path):
#####################
#S***.#************G#
#.##****##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################


RUNNING TEST: Blocked Goal Map (Unreachable)
Start Position (S): (1, 1)
Goal Position  (G): 